# Preliminaries: Functions for Exact Diagonalization

In [2]:
using LinearAlgebra
using SparseArrays #Sparse arrays are arrays that contain enough zeros that storing them in a special data structure leads to savings in space and execution time, compared to dense arrays.

#Sparse Matrix representations for local operators (definition as sparse guarantes that Kron() returns also sparse):
σx = sparse([0 1; 1 0])
σy = sparse([0 -1im; 1im 0])
σz = sparse([1 0; 0 -1])
Identity = sparse([1 0; 0 1]) #or Using LinearAlgebra and Matrix{Float64}(I, 2, 2)
σplus = (σx +1im*σy)/2
σminus = adjoint(σplus)

"""
Builds O_j as I⊗...⊗I⊗O⊗I...⊗I for a system of N sites.

Note: If the local representation is an sparse matrix, the output of this function will also be an sparse matrix. This is very important for larger N.

**Parameters:**
* `N (Int64):` System size.
* `j (Int64):` Site where the local operator O is acting.
* `matrix (Matrix{Any}):` Matrix representation for the local operator O. 

**Return:**
* `M (Matrix{Any}):` Global operator of O_j i.e. Tensor product I⊗...⊗I⊗O⊗I...⊗I.
"""
function Enlarge_Matrix_site_j(N::Int64, j::Int64, matrix)

    global Identity
    
    M = Identity
    j == 1 ? M = matrix : nothing
    
    for i=2:N 
        i == j ? M = sparse(kron(matrix, M)) :  M = sparse(kron(Identity, M)) #It is more efficient to save everything as sparse matrices. 
    end

    return M
end

"""
Builds A_iB_j as I⊗...⊗I⊗B⊗I...⊗I⊗A⊗I⊗I...⊗I (for i<j) for a system of N sites.

Note: If the local representation are sparse matrices, the output of this function will also be an sparse matrix. This is very important for larger N.

**Parameters:**
* `N (Int64):` System size.
* `i (Int64):` Site where the local operator A is acting.
* `j (Int64):` Site where the local operator B is acting.
* `matrix_i (Int64):` Matrix representation for the local operator A.
* `matrix_j (Int64):` Matrix representation for the local operator B.

**Return:**
* `M (Matrix{Any}):` Global operator of A_iB_j i.e. Tensor product I⊗...⊗I⊗B⊗I...⊗I⊗A⊗I⊗I...⊗I (for i<j).
"""
function Enlarge_Matrix_i_Matrix_j(N::Int64, i::Int64, j::Int64, matrix_i, matrix_j)
    global Identity
    
    M = Identity

    j == 1 ? M = matrix_j : nothing
    i == 1 ? M = matrix_i : nothing
 
    for k=2:N 
        if k == j
            M = sparse(kron(matrix_j, M))
        elseif k == i
            M = sparse(kron(matrix_i, M))
        else
            M = sparse(kron(Identity, M))     
        end
    end

    return M
end

"""
Builds the creation operator C†_j as an sparse matrix based on the JW transformation C†_j = (∑_i<j σz_i) σ-_j.

**Parameters:**
* `N (Int64):` System size.
* `j (Int64):` Site where creation operator is acting.

**Return:**
* `Matrix (SparseMatrixCSC{ComplexF64, Int64}):` Matrix representation of C†_j.
"""
function Build_Cdag_Matrix(N::Int64, j::Int64)
    global σminus, σz, Identity

    if j == 1
        Matrix = σminus
    else
        Matrix = σz
    end

    for i=2:N 
        if i <= j-1
        Matrix = sparse(kron(σz, Matrix))
        elseif i == j
        Matrix = sparse(kron(σminus, Matrix))
        else
        Matrix = sparse(kron(Identity, Matrix))  
        end
    end 

    return Matrix

end

"""
Builds the annihilation operator C_j as an sparse matrix based on the JW transformation C_j = (∑_i<j σz_i) σ+_j.

**Parameters:**
* `N (Int64):` System size.
* `j (Int64):` Site where annihilation operator is acting.

**Return:**
* `Matrix (SparseMatrixCSC{ComplexF64, Int64}):` Matrix representation of C_j.
"""
function Build_C_Matrix(N::Int64, j::Int64)
    global σplus, σz, Identity

    if j == 1
        Matrix = σplus
    else
        Matrix = σz
    end
    
    for i=2:N 
        if i <= j-1
        Matrix = sparse(kron(σz, Matrix))
        elseif i == j
        Matrix = sparse(kron(σplus, Matrix))
        else
        Matrix = sparse(kron(Identity, Matrix))
        end
    end 

    return Matrix

end

"""
Builds a hopping operator C†_iC_j as an sparse matrix based on the JW transformation. 

**Parameters:**
* `N (Int64):` System size.
* `i (Int64):` Site where creation operator is acting.
* `j (Int64):` Site where annihilation operator is acting.

**Return:**
* `Matrix (SparseMatrixCSC{ComplexF64, Int64}):` Matrix representation of C_j.
"""
function Build_C_dagi_Cj_Matrix(N::Int64, i::Int64, j::Int64)
    #In this case some JW strings cancel between them. C†_iC_j = σ-_i (Π_i<k<j σz_k) σ+_j (this works exactly the same for i<j and j<i because σ-*σz=σ- and σz*σ+=σ+).
    
    global σminus, σplus, σz, Identity

    if i == j #Particle number operator i.e. C†_iC_i = σ-_i σ+_i
        return Enlarge_Matrix_site_j(N, j, σminus*σplus)
    end  
    
    if i > j #C†_iC_j = (C†_jC_i)†
        return adjoint(Build_C_dagi_Cj_Matrix(N, j, i))
    end

    Matrix = Identity
    i == 1 ? Matrix = σminus : nothing
 
    for k=2:N # i < j case
        if k == i
            Matrix = sparse(kron(σminus, Matrix))
        elseif k == j
            Matrix = sparse(kron(σplus, Matrix))
        elseif k > i && k < j
            Matrix = sparse(kron(σz, Matrix))       
        else
            Matrix = sparse(kron(Identity, Matrix))      
        end
    end

    return Matrix
end

Build_C_dagi_Cj_Matrix

# Hamiltonian and Lindbladian construction:

In [3]:
function Build_H_interacting(ε, ts, U)
    #ε: on-site energies
    #ts: hopping
    #U: interaction

    D = length(ε)
    H = zeros(2^D, 2^D)

    for i=1:D
        H += ε[i]*Build_C_dagi_Cj_Matrix(D, i, i)
    end
    
    for i=1:D-1 
        H += -ts*Build_C_dagi_Cj_Matrix(D, i, i+1)
        H += -ts*Build_C_dagi_Cj_Matrix(D, i+1, i)
        H += U*Build_C_dagi_Cj_Matrix(D, i, i)*Build_C_dagi_Cj_Matrix(D, i+1, i+1)
    end

    return H
end

function custom_log(x, base::Real)
    return log(x)/log(base)
end

function Build_Liouvillian_Superoperator(H, L_k_list, local_dimension)

    #l_k has inside the sqrt(γ_K)

    Identity = Diagonal(ones(local_dimension))
    N = custom_log(size(H)[1], local_dimension)    
    superIdentity = Diagonal(ones(Int64(local_dimension^N)))
    
    superH = -1im*(kron(superIdentity, H) - kron(transpose(H), superIdentity))
    superL = spzeros(ComplexF64, size(superH))
    
    for k=1:length(L_k_list)
        L_k = L_k_list[k]
        superL = superL + kron(conj(L_k), L_k) -0.5*(kron(superIdentity, adjoint(L_k)*L_k) + kron(transpose(L_k)*conj(L_k), superIdentity)) 
    end

    Liouvillian_Superoperator = superH + superL
    return Liouvillian_Superoperator*1im #I added this 1im in order to have Schrodinger-type equation
end

function H(t)
    global ε_t_array, ts, U
    ε_s = [ε_t(t) for ε_t = ε_t_array]
    H_t = Build_H_interacting(ε_s, ts, U)
    return H_t
end

function L(t)
    global L_k_list

    H_t = H(t)
    L_op = Build_Liouvillian_Superoperator(H_t, L_k_list, 2)
    return L_op
end    

L (generic function with 1 method)

# Starting point:
We have a time dependent system:

$$ \hat{H}_{S}(t) = \sum_{i=1}^{D} \varepsilon_{i}(t) \hat{c}_{i}^{\dagger}\hat{c}_{i} +\sum_{i = 1}^{D-1} [- t_{s}(\hat{c}_{i+1}^{\dagger}\hat{c}_{i} + h.c.) + U \hat{n}_{i}\hat{n}_{i+1}] $$

The on-site energies are tilted:

$$ \varepsilon_j = E \left(j - \frac{D+1}{2} \right)$$

And under the action of a laser they become time dependent functions: 

$$\varepsilon_{i}(t) = \varepsilon_{i} \quad , \quad \varepsilon_{j}(t) = \varepsilon_{j}E(t)$$

with $D =4$, $i = \{ 1,4 \}$ and $j = \{2, 3\}$.

**Note:** As our hamiltonian is a time dependent function, our observables will never become steady, we will not have convergence, at least not common convergence. We will have cycle-average converged quantities, i.e. for all observables you need to see that the values are oscillating and calculate the average over several periods of time.

In [4]:
global ε_t_array, ts, U

ts = -1.0
D = 4
E = 6*abs(ts)/(D-1)
U = 0.0*abs(ts)
ε_system = Float64[E*(j-(D+1)/2) for j=1:D]

f_S = [0.0 for j=1:D]
γ_S = [0.0 for j=1:D]

f_S[1] = 1.0
f_S[D] = 0.0
γ_S[1] = 1.0
γ_S[D] = 0.0

L_k_list = []

for i = 1:D
    γ = γ_S[i]
    f = f_S[i]
    push!(L_k_list, sqrt(γ*(1-f))*Build_C_Matrix(D, i))
    push!(L_k_list, sqrt(γ*f)*Build_Cdag_Matrix(D, i))
end

The idea of this project is to study the effect of the phase in the time driving. Let's start with

$$ E(t) = cos(\omega t  + \phi) $$

You should perform the time evolution and find the cycle-average currents.

In [15]:
I_matrix = Matrix(Enlarge_Matrix_site_j(D, 1, Diagonal(ones(2))))
I_vec = Matrix(reshape(I_matrix, (2^(2*D), 1)))

ρ0_vec = reshape(I_matrix, (2^(2*D), 1))
ρ0_vec = ρ0_vec/(adjoint(I_vec)*ρ0_vec);

As a warm up take $\phi = 0$ and $\omega \in (0.5, 8)$ and plot $J_{i \rightarrow i + 1}^{P}$ vs $\omega$

In [ ]:
ω = 4.4
ϕ = 0.0

ε_1(t::Float64) = ε_system[1]  
ε_2(t::Float64) = ε_system[2]*cos(ω*t + ϕ) 
ε_3(t::Float64) = ε_system[3]*cos(ω*t + ϕ) 
ε_4(t::Float64) = ε_system[4]     
ε_t_array = [ε_1, ε_2, ε_3, ε_4]

dt = 0.05
Numsteps = 2000 #13 seconds / 100 steps.

times = [0.0]
ρ_vec = ρ0_vec;

for n =1:Numsteps 
    t = times[end]
    U_op = exp(-1im*Matrix(L(t))*dt)
    ρ_vec = U_op*ρ_vec
    ρ_vec = ρ_vec/(adjoint(I_vec)*ρ_vec)

    #Here you need to implement the calculation of the observables.
    #Do reshape ρ_vec -> ρ_matrix
    #Define operator A
    #Calculate observable <A> = Tr(Aρ)
    #Add <A> to some array or data structure that you can use later for plotting.
    
    push!(times, t+dt)
end

During this project you should study the following questions:

- How do you determine if you have convergence in your quantities?
- What is the period of the observables? Consider expectation values of $\hat{n}_{i}$ and $\hat{J}_{i \rightarrow i + 1}^{P} = - i t_{s}( \hat{c}_{i}^{\dagger}\hat{c}_{i+1}  - \hat{c}_{i+1}^{\dagger}\hat{c}_{i} ) $.
- What is the impact of $\phi$ in the cycle average currents (take $\omega$ where the $J_{i \rightarrow i + 1}^{P} $ has a maxima)? If there is an interesting impact find a smart way to plot $J_{i \rightarrow i + 1}^{P}$ vs $\omega$ vs $\phi$.
- Is there rectification if you consider Forward and Reverse bias? How is affected by $\phi$. If there is no rectification, then you should do the simulation within the ML framework (Do not worry, all of that is already implemented).
- Repeat the analysis but now using pulses $E(t) = e^{-(t -t_{0})/2\sigma^2} cos(\omega t + \phi)$.
- Consider $E(t) = cos(\omega t )$ and analyze what happens if you consider different frequencies for sites 2 and 3. Plot $J_{i \rightarrow i + 1}^{P}$ vs $\omega_{2}$ vs $\omega_{3}$.

At the end you should write a report. Every week upload your advances to this GitHub repository.

**Example for calculating observables:** $\langle n_{1} \rangle$

In [14]:
ρ0_matrix = reshape(ρ0_vec, (2^D, 2^D)) # First you need to reshape the density matrix back to its vector form.
OP = Build_C_dagi_Cj_Matrix(D, 1, 1) #Define the operator
Observable = tr(OP*ρ0_matrix)/tr(ρ0_matrix) #<A> = Tr(Aρ). It is good to divide by Tr(ρ), this is usually 1 but is better to do it just in case.

0.5 + 0.0im